<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2025 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用LoRA微調Keras中的Gemma

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/core/lora_tuning"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
<td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/docs/core/lora_tuning.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemini/gemma-cookbook/blob/main/docs/core/lora_tuning.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemini%2Fgemma-cookbook%2Fmain%2Fdocs%2Fcore%2Flora_tuning.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemini/gemma-cookbook/blob/main/docs/core/lora_tuning.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

Gemma 等生成式人工智慧 (AI) 模型可有效執行各種任務。您可以使用特定領域的資料進一步微調 Gemma 模型以執行情緒分析等任務。然而，透過更新數十億個參數來產生完整的fine-tuning 生成模型是資源密集型的，需要專門的硬體（例如 GPU）、處理時間和記憶體來載入模型參數。
[低階適應](https://arxiv.org/abs/2106.09685) (LoRA) 是一種fine-tuning 技術，透過凍結模型的權重並向模型中插入較少數量的新權重，大大減少了下游任務的可訓練參數的數量。這項技術使得 LoRA 的訓練速度更快、記憶體效率更高，並產生更小的模型權重（幾百 MB），同時保持模型輸出的品質。本教學將引導您使用Keras 在Gemma 模型上執行LoRA fine-tuning 模型。

## 設定

要完成本教學，您首先需要完成 [Gemma 設定](https://ai.google.dev/gemma/docs/setup) 中的設定說明。 Gemma 設定說明向您展示如何執行以下操作：
* 在 [kaggle.com](https://kaggle.com) 上造訪Gemma。
* 選擇具有足夠資源的 Colab runtime 進行調整
您要執行的 Gemma 模型。 [了解更多](https://ai.google.dev/gemma/docs/core#sizes)。* 產生並設定 Kaggle 使用者名稱和 API 金鑰。

完成 Gemma 設定後，請前往下一部分，您將為 Colab 環境設定環境變數。

### 選擇 Colab runtime

要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 Gemma 模型。在這種情況下，您可以使用 T4 GPU：
1. 在Colab視窗的右上角，選擇&#9662; （**附加連線選項**）。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **T4 GPU**。

### 設定您的 API 金鑰

若要使用 Gemma，您必須提供 Kaggle 使用者名稱和 Kaggle API 金鑰。
若要產生 Kaggle API 金鑰，請前往 Kaggle 使用者個人資料的 **帳戶** 選項卡，然後選擇 **建立新 token**。這將觸發包含您的 API 憑證的 `kaggle.json` 檔案的下載。
在 Colab 中，選擇左側窗格中的 **Secrets** (🔑)，然後新增您的 Kaggle 使用者名稱和 Kaggle API 金鑰。將您的使用者名稱儲存在名稱`KAGGLE_USERNAME` 下，將您的API 金鑰儲存在名稱`KAGGLE_KEY` 下。

### 設定環境變數

設定`KAGGLE_USERNAME` 和`KAGGLE_KEY` 的環境變數。

In [ ]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### 安裝 Keras 軟體包

安裝 Keras 和 KerasHub Python 軟體包。

In [ ]:
!pip install -q -U keras-hub
!pip install  -q -U keras

### 選擇後端

Keras 是高級、多framework 深度學習API，設計簡單易用。使用Keras 3，您可以在三個後端之一上執行工作流程：TensorFlow、JAX 或PyTorch。在本教學中，為 JAX 設定後端，因為它通常可以提供更好的效能。

In [ ]:
os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

### 導入包

導入本教學所需的Python包，包括Keras和KerasHub。

In [ ]:
import keras
import keras_hub

## 負載模型

Keras 提供Gemma 和許多其他流行的[模型架構](https://keras.io/keras_hub/api/models/) 的實作。使用 `Gemma3CausalLM.from_preset()` 方法為因果語言建模設定端對端 Gemma 模型。因果語言模型根據前一個 tokens 預測下一個 token。

In [ ]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")
gemma_lm.summary()

`Gemma3CausalLM.from_preset()` 方法根據預設的架構和權重實例化模型。在上面的程式碼中，字串`"gemma#_xxxxxxx"`指定Gemma的預設版本和參數大小。您可以在 [Kaggle](https://www.kaggle.com/models/keras/gemma3) 上的 **型號變體** 清單中找到 Gemma 型號的程式碼字串。

## 微調前的推論

下載並設定 Gemma 模型後，您可以使用各種 prompt 查詢它，看看它如何回應。

### 歐洲之旅prompt

查詢模型以取得有關去歐洲旅行時應該做什麼的建議。

In [ ]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

prompt = template.format(
    instruction="What should I do on a trip to Europe?",
    response="",
)
sampler = keras_hub.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)
print(gemma_lm.generate(prompt, max_length=256))

Instruction:
What should I do on a trip to Europe?

Response:
The first thing to know is that you will have a great time!

Europe is a great place for a vacation. The countries of Europe are all very different and offer a wide range of activities and attractions. The countries of Europe are also very close to each other, which means you can visit many different places within a short time.

The best way to plan a trip to Europe is to look up the countries you want to visit and see what activities are offered in each country. You can also look for tours and tours that offer a good value for money.

You can also look for hotels and flights that offer good deals. If you are looking for a good value for money, you should look for hotels and flights that offer good deals. This means you will have a great time on your trip!

The next step is to book your tickets to the countries you want to visit. If you are planning to visit many countries, it's a good idea to book your tickets early. This m

模型會給出有關如何計劃旅行的一般提示。

### 光合作用prompt

提示模型以 5 歲兒童能夠理解的簡單術語解釋光合作用。

In [ ]:
prompt = template.format(
    instruction="Explain the process of photosynthesis in a way that a child could understand.",
    response="",
)
print(gemma_lm.generate(prompt, max_length=256))

Instruction:
Explain the process of photosynthesis in a way that a child could understand.

Response:
Photosynthesis is a biological process that occurs in plants, algae, and some other organisms. In the process, light energy is captured and converted into the energy stored in the bonds of organic molecules. The process is crucial for life on Earth because it enables plants to use carbon dioxide and water to produce glucose and oxygen, which are essential for all living things.
The process involves several stages:
1. Light Reactions: Light energy is absorbed by pigments in the chloroplasts of the plant, converting it into chemical energy in the form of ATP and reducing power.
2. Carbon Fixation: During this stage, carbon dioxide is combined with hydrogen to form organic molecules such as starch or glucose, which are used as a source of energy.
3. Calvin Cycle: The process of carbon fixation occurs in the stroma of the chloroplasts. It involves the capture and reduction of carbon dioxid

模型回應包含兒童可能不容易理解的單字，例如葉綠素。

## LoRA fine-tuning

本節向您展示如何使用低階適應 (LoRA) 調整技術來執行fine-tuning。這種方法允許您使用更少的計算資源來更改 Gemma 模型的行為。

### 加載dataset

如果要與 Keras `fit()` fine-tuning 方法一起使用，請下載現有 dataset 並格式化來準備 dataset 進行調整。本教學使用 [Databricks Dolly 15k dataset](https://huggingface.co/datasets/databricks/databricks-dolly-15k) 為fine-tuning。 dataset 包含 15,000 個高品質的人工生成的 prompt 和專門為調整生成模型而設計的響應對。

In [ ]:
!wget -O databricks-dolly-15k.jsonl https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl

--2025-04-10 20:48:49--  https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl
Resolving huggingface.co (huggingface.co)... 3.163.189.37, 3.163.189.114, 3.163.189.74, ...
Connecting to huggingface.co (huggingface.co)|3.163.189.37|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs.hf.co/repos/34/ac/34ac588cc580830664f592597bb6d19d61639eca33dc2d6bb0b6d833f7bfd552/2df9083338b4abd6bceb5635764dab5d833b393b55759dffb0959b6fcbf794ec?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27databricks-dolly-15k.jsonl%3B+filename%3D%22databricks-dolly-15k.jsonl%22%3B&Expires=1744321729&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0NDMyMTcyOX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy8zNC9hYy8zNGFjNTg4Y2M1ODA4MzA2NjRmNTkyNTk3YmI2ZDE5ZDYxNjM5ZWNhMzNkYzJkNmJiMGI2ZDgzM2Y3YmZkNTUyLzJkZjkwODMzMzhiNGFiZDZiY2ViNTYzNTc2NGRhYjVkODMzYjM5M2I1NTc1OWRmZmIw

### 格式化調整數據

格式化下载的数据以用于 Keras `fit()` 方法。以下代码提取训练示例的子集以更快地执行notebook。考慮使用更多訓練資料來獲得更高品質fine-tuning。

In [ ]:
import json

prompts = []
responses = []
line_count = 0

with open("databricks-dolly-15k.jsonl") as file:
    for line in file:
        if line_count >= 1000:
            break  # Limit the training examples, to reduce execution time.

        examples = json.loads(line)
        # Filter out examples with context, to keep it simple.
        if examples["context"]:
            continue
        # Format data into prompts and response lists.
        prompts.append(examples["instruction"])
        responses.append(examples["response"])

        line_count += 1

data = {
    "prompts": prompts,
    "responses": responses
}

### 設定LoRA調諧

使用Keras `model.backbone.enable_lora()` 方法啟動LoRA 調諧，包括LoRA 等級值。 *LoRA 等級* 決定加到 LLM 原始權重的可訓練矩陣的維數。它控制 fine-tuning 調整的表現力和精確度。更高的排名意味著可以進行更詳細的更改，但也意味著更多可訓練的參數。較低的等級意味著較少的計算開銷，但可能不太精確的適應。
此範例使用 LoRA 等級 4。在實踐中，從相對較小的等級開始（例如 4、8、16）。此設定對於實驗而言計算效率較高。使用此排名訓練您的模型並評估您的任務的表現改進。在後續試驗中逐漸提高排名，看看是否會進一步提高效能。

In [ ]:
# Enable LoRA for the model and set the LoRA rank to 4.
gemma_lm.backbone.enable_lora(rank=4)

設定LoRA等級後檢查模型摘要。請注意，與模型中的參數總數相比，啟用 LoRA 會顯著減少可訓練參數的數量：

In [ ]:
gemma_lm.summary()

設定fine-tuning設定的其餘部分，包括預處理器設定、最佳化器、調整時期數和批次大小：

In [ ]:
# Limit the input sequence length to 256 (to control memory usage).
gemma_lm.preprocessor.sequence_length = 256
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

### 執行微調過程

使用`fit()` 方法執行fine-tuning 進程。此過程可能需要幾分鐘時間，具體取決於您的運算資源、資料大小和紀元數：

In [ ]:
gemma_lm.fit(data, epochs=1, batch_size=1)

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 923s 888ms/step - loss: 1.5586 - sparse_categorical_accuracy: 0.5251


#### NVIDIA GPU 上的混合精準度 fine-tuning

建議fine-tuning 使用全精度。在 NVIDIA GPU 上fine-tuning 時，您可以使用混合精度 (`keras.mixed_precision.set_global_policy('mixed_bfloat16')`) 來加速訓練，同時對訓練品質的影響最小。

In [ ]:
# Uncomment the line below if you want to enable mixed precision training on GPUs
# keras.mixed_precision.set_global_policy('mixed_bfloat16')

## fine-tuning之後的推論

在 fine-tuning 之後，當為調整後的模型提供相同的 prompt 時，您應該會看到響應發生變化。

### 歐洲之旅prompt

嘗試之前的歐洲之旅 prompt 並注意回應中的差異。

In [ ]:
prompt = template.format(
    instruction="What should I do on a trip to Europe?",
    response="",
)
sampler = keras_hub.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)
print(gemma_lm.generate(prompt, max_length=256))

Instruction:
What should I do on a trip to Europe?

Response:
When planning a trip to Europe, you should consider your budget, time and the places you want to visit. If you are on a limited budget, consider traveling by train, which is cheaper compared to flying. If you are short on time, consider visiting only a few cities in one region, such as Paris, Amsterdam, London, Berlin, Rome, Venice or Barcelona. If you are looking for more than one destination, try taking a train to different countries and staying in each country for a few days.


該模型現在對有關訪問歐洲的問題提供了更簡短的答案。

### 光合作用prompt

嘗試之前的光合作用解釋 prompt 並注意反應中的差異。

In [ ]:
prompt = template.format(
    instruction="Explain the process of photosynthesis in a way that a child could understand.",
    response="",
)
print(gemma_lm.generate(prompt, max_length=256))

Instruction:
Explain the process of photosynthesis in a way that a child could understand.

Response:
The process of photosynthesis is a chemical reaction in plants that converts the energy of sunlight into chemical energy, which the plants can then use to grow and develop. During photosynthesis, a plant will absorb carbon dioxide (CO2) from the air and water from the soil and use the energy from the sun to produce oxygen (O2) and sugars (glucose) as a by-product.


該模型現在用更簡單的術語解釋了光合作用。

## 改善微調結果

出於演示目的，本教學僅在一個時期內對 dataset 的一小部分子集上的模型進行微調，並使用較低的 LoRA 等級值。為了從微調模型中獲得更好的反應，您可以嘗試：
1. 增加 fine-tuning dataset 的大小
2. 訓練更多步驟（epochs）
3. 設定更高的LoRA等級
4. 修改`learning_rate`和`weight_decay`等超參數值。

## 摘要與後續步驟

本教學介紹了使用Keras 在Gemma 模型上的LoRA fine-tuning。接下來查看以下文檔：
* 了解如何[使用 Gemma 模型產生文字](https://ai.google.dev/gemma/docs/get_started)。
* 了解如何[在Gemma 模型上執行分佈式fine-tuning 和inference](https://ai.google.dev/gemma/docs/core/distributed_tuning)。
* 了解如何[使用 Gemma 和 Vertex AI 開放式模型](https://cloud.google.com/vertex-ai/docs/generative-ai/open-models/use-gemma)。
* 了解如何[使用Keras 微調Gemma 並部署至Vertex AI](https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/model_garden/model_garden_gemma_kerasnlp_to_vertexai.ipynb)。